# ChaoSSM — Phase 0: Premise Test

**Purpose.** Before committing ~3,000 GPU-hours to the full study, establish that the
central mechanism is even viable. Phase 0 answers four questions, in order. A failure at
any one of them changes the project.

| # | Check | Question | Failure means |
|---|-------|----------|---------------|
| 1 | **Trainability** | Does truncated BPTT through a learnable chaotic map converge without gradient explosion? | Backprop-through-chaos is not tractable as designed; `k` or the clipping scheme must change. |
| 2 | **Corridor controllability** | Where does λ̂ *actually* sit during training? | The asymmetric hinge is a barrier, not an attractor — it may be satisfied without ever operating near criticality. |
| 3 | **Estimator validity** | Does the cheap power-iteration λ̂ track the true Benettin λ₁? And does λ of φ track λ of the *full recurrence*? | The regularizer is steering a biased or wrong quantity. |
| 4 | **Non-degeneracy** | Is the learned map genuinely chaotic (D_KY > 1), or has it collapsed to near-linear / a fixed point? | "Chaos" is decorative; ChaoSSM reduces to a nonlinear SSM. |

**The specific risk this notebook was built around.** The proposed regularizer

```
L_Lyap = α·max(0, λ̂ − 0.15)² + β·max(0, −0.05 − λ̂)²
```

is exactly **zero** everywhere inside [−0.05, +0.15]. It provides no gradient to sit anywhere
in particular within the corridor — it only pushes back at the edges. The LM objective is then
free to drive λ̂ to wherever optimization is easiest, which is almost certainly the stable end.
If that happens, the constraint is satisfied while the model never operates near criticality,
and H₅ becomes untestable because the corridor arm and the gradient-flossed arm both end up
near λ≈0 in practice. This has the same shape as the BCM risk collapsing to its trivial
minimum. **Check 2 therefore logs the full distribution of λ̂ over training, not just whether
the constraint was satisfied.**

**Three arms** are trained so the question is answerable:
- `corridor` — the proposed asymmetric hinge
- `floss` — gradient flossing's symmetric λ̂² (the H₅ comparison)
- `free` — no Lyapunov regularization (the control that reveals what the LM loss wants on its own)

If `corridor` ≈ `free`, the regularizer is not doing work.

## Setup

In [6]:
import os, json, math, time, itertools, warnings
from dataclasses import dataclass, asdict, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

torch 2.11.0+cu128 | device: cuda
gpu: Tesla T4


In [7]:
# --- Results directory: Google Drive if in Colab, else local -------------------
def setup_results_dir(project="ChaoSSM", phase="phase0"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab (or Drive unavailable: {type(e).__name__}). "
              f"Falling back to local dir.")
    os.makedirs(base, exist_ok=True)
    print("results ->", base)
    return base

RESULTS = setup_results_dir()
FIGDPI = 300

def savefig(fig, name):
    path = os.path.join(RESULTS, name)
    fig.savefig(path, dpi=FIGDPI, bbox_inches="tight")
    print("saved", path)

def savetable(df, name):
    path = os.path.join(RESULTS, name)
    df.to_csv(path, index=False)
    print("saved", path)

def savejson(obj, name):
    path = os.path.join(RESULTS, name)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    print("saved", path)

Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase0


## Configuration

**Run `SMOKE_TEST = True` first.** It exercises every code path in ~2 minutes so that a
crash or a silently-wrong estimator shows up before you spend an hour on the real run.
Only then set it to `False`.

On seed count: paired Wilcoxon with n=5 has a two-sided minimum achievable p of 0.0625 — it
*cannot* reach significance at α=0.05 no matter how clean the effect. n=8 gives a floor of
0.0078. The default is 8 for that reason.

In [8]:
SMOKE_TEST = True   # <-- set False for the real run after this passes

@dataclass
class Config:
    # model
    state_dim: int = 48          # N
    d_model: int = 48
    vocab: int = 16
    # task (copying)
    copy_len: int = 16
    # training
    steps: int = 600
    batch: int = 64
    lr: float = 3e-4
    bptt_k: int = 64             # truncation horizon
    grad_clip: float = 1.0
    lyap_alpha: float = 0.01
    lam_low: float = -0.05
    lam_high: float = 0.15
    # instrumentation
    lyap_every: int = 10         # estimate lambda_hat every N steps
    lyap_subsample: int = 8      # batch elements used for the Jacobian estimate
    # analysis
    benettin_transient: int = 100
    benettin_measure: int = 300
    seeds: int = 8
    arms: tuple = ("corridor", "floss", "free")

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 60
    CFG.seeds = 2
    CFG.benettin_measure = 60
    CFG.benettin_transient = 20
    print("*** SMOKE TEST — results are NOT meaningful, this only checks the pipeline runs ***")

print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

*** SMOKE TEST — results are NOT meaningful, this only checks the pipeline runs ***
{
  "state_dim": "48",
  "d_model": "48",
  "vocab": "16",
  "copy_len": "16",
  "steps": "60",
  "batch": "64",
  "lr": "0.0003",
  "bptt_k": "64",
  "grad_clip": "1.0",
  "lyap_alpha": "0.01",
  "lam_low": "-0.05",
  "lam_high": "0.15",
  "lyap_every": "10",
  "lyap_subsample": "8",
  "benettin_transient": "20",
  "benettin_measure": "60",
  "seeds": "2",
  "arms": "('corridor', 'floss', 'free')"
}


## 1. Model

The chaotic map and cell follow §4.2–4.5 of the proposal. Two things worth flagging in the
implementation:

**The state domain.** `φ` returns a sigmoid, so its output lies in (0,1) — the domain the
logistic map needs. But the additive input term `B x_t` can push the state outside that range,
and the `clamp` inside `φ` then silently absorbs it. How often that clamp is active is itself
diagnostic, so we count it.

**Which Jacobian.** §4.4 regularizes `J_φ`, the Jacobian of the chaotic map alone. But the
dynamical behavior that actually governs memory and gradient flow is the Jacobian of the
*full recurrence*:

```
h_t = g_t ⊙ φ(A h_{t-1}) + (1 − g_t) ⊙ h_{t-1} + B x_t
J_full = diag(g_t) · J_φ(A h_{t-1}) · A + diag(1 − g_t)
```

Gating and `A` sit between the regularized quantity and the real one. If λ(φ) and λ(full)
diverge, the regularizer is steering the wrong target — that is part of Check 3.

In [9]:
class LogisticChaoticMap(nn.Module):
    '''phi_theta from proposal S4.3, Option A.'''
    def __init__(self, state_dim):
        super().__init__()
        self.w_r = nn.Parameter(torch.zeros(state_dim))
        W = torch.empty(state_dim, state_dim)
        nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        self.register_buffer("clamp_hits", torch.zeros(()))
        self.register_buffer("clamp_total", torch.zeros(()))

    def get_r(self):
        # r in [3.57, 4.0] -> guaranteed chaotic regime for the scalar map
        return 3.57 + 0.43 * torch.sigmoid(self.w_r)

    def forward(self, h, count_clamp=False):
        if count_clamp:
            with torch.no_grad():
                out = ((h < 1e-6) | (h > 1 - 1e-6)).float()
                self.clamp_hits += out.sum()
                self.clamp_total += out.numel()
        h_c = torch.clamp(h, 1e-6, 1 - 1e-6)
        r = self.get_r()
        lm_out = r * h_c * (1 - h_c)
        return torch.sigmoid(h_c @ self.W_mix.T + lm_out)


class ChaoSSMCell(nn.Module):
    '''Single ChaoSSM recurrence (proposal S4.5).'''
    def __init__(self, state_dim, d_model):
        super().__init__()
        self.N, self.d = state_dim, d_model
        self.phi = LogisticChaoticMap(state_dim)
        # A initialized diagonal ~ U[0.9, 1.1] per S4.2
        A0 = torch.diag(torch.empty(state_dim).uniform_(0.9, 1.1))
        self.A = nn.Parameter(A0)
        self.B = nn.Parameter(torch.randn(state_dim, d_model) * (1.0 / math.sqrt(d_model)))
        self.W_g = nn.Parameter(torch.randn(state_dim, d_model) * (1.0 / math.sqrt(d_model)))
        self.b_g = nn.Parameter(torch.zeros(state_dim))

    def step(self, h, x, count_clamp=False):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        pre = h @ self.A.T
        return g * self.phi(pre, count_clamp) + (1 - g) * h + x @ self.B.T

    def init_state(self, batch, device):
        # start mid-domain, where the logistic map is well conditioned
        return torch.full((batch, self.N), 0.5, device=device)


class ChaoSSMLM(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(cfg.vocab + 2, cfg.d_model)  # +2 for SEP/PAD
        self.cell = ChaoSSMCell(cfg.state_dim, cfg.d_model)
        self.readout = nn.Linear(cfg.state_dim, cfg.vocab + 2)

    def forward(self, tokens, return_traj=False, count_clamp=False):
        B, T = tokens.shape
        x = self.emb(tokens)                       # B,T,d
        h = self.cell.init_state(B, tokens.device)
        outs, traj = [], []
        for t in range(T):
            h = self.cell.step(h, x[:, t], count_clamp)
            outs.append(self.readout(h))
            if return_traj:
                traj.append(h.detach())
            # truncated BPTT
            if (t + 1) % self.cfg.bptt_k == 0:
                h = h.detach()
        logits = torch.stack(outs, dim=1)
        if return_traj:
            return logits, torch.stack(traj, dim=1)
        return logits

## 2. Lyapunov estimators

Two estimators, deliberately kept separate:

- **`lambda_power`** — the cheap per-step estimate the *regularizer* uses (power iteration on
  `JᵀJ`, k=5), exactly as specified in §6.3.
- **`benettin_spectrum`** — the full spectrum via QR reorthogonalization, used only for
  *post-hoc validation*. This is the reference the cheap estimator is checked against.

`D_KY` (Kaplan–Yorke dimension) is computed from the Benettin spectrum.

In [10]:
from torch.func import jacrev, vmap

def lambda_power(fn, h_batch, k=5):
    '''Cheap MLE estimate: log of largest singular value of J_fn at each point.
    This is what the regularizer sees (proposal S6.3).'''
    J = vmap(jacrev(fn))(h_batch)                        # (B,N,N)
    v = torch.randn(J.shape[0], J.shape[1], 1, device=J.device, dtype=J.dtype)
    for _ in range(k):
        v = torch.bmm(J.transpose(1, 2), torch.bmm(J, v))
        v = v / (v.norm(dim=1, keepdim=True) + 1e-8)
    sigma_max = torch.sqrt((torch.bmm(J, v) ** 2).sum(dim=1)).squeeze(-1)
    return torch.log(sigma_max + 1e-8)


@torch.no_grad()
def benettin_spectrum(step_fn, h0, inputs, N, T_transient, T_measure):
    '''Full Lyapunov spectrum via QR reorthogonalization (proposal S5.3, Analysis 1).
    step_fn(h, x) -> h_next, operating on a single (N,) state.'''
    device = h0.device
    h = h0.clone()
    for t in range(T_transient):
        h = step_fn(h, inputs[t % len(inputs)])
    Q = torch.eye(N, device=device, dtype=h.dtype)
    lyap = torch.zeros(N, device=device, dtype=h.dtype)
    for t in range(T_measure):
        x = inputs[(T_transient + t) % len(inputs)]
        J = jacrev(lambda hh: step_fn(hh, x))(h)          # (N,N)
        Q = J @ Q
        Q, R = torch.linalg.qr(Q)
        d = torch.diagonal(R)
        lyap += torch.log(torch.abs(d) + 1e-12)
        h = step_fn(h, x)
    return (lyap / T_measure).sort(descending=True).values


def kaplan_yorke(spectrum):
    '''D_KY from a sorted-descending Lyapunov spectrum.'''
    s = spectrum.detach().cpu().numpy().astype(np.float64)
    csum = np.cumsum(s)
    idx = np.where(csum >= 0)[0]
    if len(idx) == 0:
        return 0.0                      # all directions contracting -> point attractor
    j = int(idx[-1]) + 1
    if j >= len(s):
        return float(len(s))            # spectrum never turns negative
    denom = abs(s[j])
    if denom < 1e-12:
        return float(j)
    return float(j + csum[j - 1] / denom)

### 2b. Estimator validation on a system with a known answer (Check 3, part 1)

Before trusting either estimator on the model, verify both against ground truth. The scalar
logistic map at r=4 has λ₁ = ln 2 ≈ 0.6931 exactly. If the estimators cannot recover that,
nothing measured later means anything.

This is the calibration-trap check: an estimator that returns a plausible-looking number on
a system whose answer you don't know is worth nothing until it has been shown to return the
*right* number on a system whose answer you do.

In [11]:
def validate_estimators():
    rows = []
    for r_val, truth, label in [(4.0, math.log(2), "r=4.0 (fully chaotic)"),
                                (3.2, None, "r=3.2 (period-2, expect lambda<0)")]:
        # scalar logistic map as a 1-D "system"
        def f(h):
            return r_val * h * (1 - h)

        # --- power iteration estimate along a trajectory
        h = torch.tensor([0.31234], dtype=torch.float64)
        traj = []
        for _ in range(2000):
            h = f(h)
            traj.append(h.clone())
        pts = torch.stack(traj[500:]).reshape(-1, 1)
        lp = lambda_power(lambda z: f(z), pts, k=5).mean().item()

        # --- Benettin estimate
        dummy = [torch.zeros(1, dtype=torch.float64)]
        sp = benettin_spectrum(lambda hh, x: f(hh),
                               torch.tensor([0.31234], dtype=torch.float64),
                               dummy, N=1, T_transient=500, T_measure=2000)
        bn = sp[0].item()

        rows.append(dict(system=label, truth=truth,
                         lambda_power=lp, lambda_benettin=bn,
                         power_err=None if truth is None else abs(lp - truth),
                         benettin_err=None if truth is None else abs(bn - truth)))
    df = pd.DataFrame(rows)
    return df

val_df = validate_estimators()
print(val_df.to_string(index=False))

tol = 0.02
row0 = val_df.iloc[0]
ok = (row0.power_err < tol) and (row0.benettin_err < tol)
print(f"\nESTIMATOR VALIDATION: {'PASS' if ok else 'FAIL'} "
      f"(tolerance {tol} vs ln2={math.log(2):.4f})")
if not ok:
    print("!! Do not trust downstream lambda values until this passes.")
savetable(val_df, "phase0_estimator_validation.csv")

                           system    truth  lambda_power  lambda_benettin  power_err  benettin_err
            r=4.0 (fully chaotic) 0.693147      0.690642         0.692591   0.002506      0.000556
r=3.2 (period-2, expect lambda<0)      NaN     -0.916291        -0.916291        NaN           NaN

ESTIMATOR VALIDATION: PASS (tolerance 0.02 vs ln2=0.6931)
saved /content/drive/My Drive/ChaoSSM/results/phase0/phase0_estimator_validation.csv


## 3. Task — copying

In [12]:
SEP = None  # filled in below

def make_copy_batch(cfg, batch, device, gen):
    '''[x_1..x_L, SEP, pad...] -> targets [ignore..., x_1..x_L]'''
    L, V = cfg.copy_len, cfg.vocab
    sep_id, pad_id = V, V + 1
    src = torch.randint(0, V, (batch, L), device=device, generator=gen)
    inp = torch.cat([src,
                     torch.full((batch, 1), sep_id, device=device),
                     torch.full((batch, L), pad_id, device=device)], dim=1)
    tgt = torch.cat([torch.full((batch, L + 1), -100, device=device), src], dim=1)
    return inp, tgt

globals()['SEP'] = CFG.vocab
print("sequence length:", CFG.copy_len * 2 + 1)

sequence length: 33


## 4. Training with instrumentation

Each arm differs *only* in the Lyapunov loss term. Everything else — data, init, seed,
optimizer, clipping — is held fixed, so any difference between arms is attributable to the
regularizer.

Recorded every `lyap_every` steps: λ̂ (of φ), gradient norm before clipping, task loss,
accuracy, learned r distribution, and clamp-activation rate.

In [13]:
def lyap_loss(lam, arm, cfg):
    if arm == "corridor":
        return cfg.lyap_alpha * (F.relu(lam - cfg.lam_high) ** 2
                                 + F.relu(cfg.lam_low - lam) ** 2)
    if arm == "floss":
        return cfg.lyap_alpha * lam ** 2
    if arm == "free":
        return torch.zeros((), device=lam.device)
    raise ValueError(arm)


def train_one(arm, seed, cfg, verbose=False):
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE); gen.manual_seed(seed)

    model = ChaoSSMLM(cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.95))

    hist = []
    diverged = False
    t0 = time.time()

    for step in range(cfg.steps):
        inp, tgt = make_copy_batch(cfg, cfg.batch, DEVICE, gen)
        logits, traj = model(inp, return_traj=True, count_clamp=True)

        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                                 tgt.reshape(-1), ignore_index=-100)

        # --- lambda_hat on a subsample of states (what the regularizer sees)
        need_lyap = (step % cfg.lyap_every == 0) or (arm != "free")
        if need_lyap:
            idx = torch.randperm(traj.shape[0], device=DEVICE)[:cfg.lyap_subsample]
            tsel = torch.randperm(traj.shape[1], device=DEVICE)[:cfg.lyap_subsample]
            pts = traj[idx][:, tsel].reshape(-1, cfg.state_dim)
            pre = pts @ model.cell.A.T.detach()
            lam = lambda_power(lambda z: model.cell.phi(z), pre).mean()
        else:
            lam = torch.zeros((), device=DEVICE)

        L_lyap = lyap_loss(lam, arm, cfg)
        L_total = L_task + L_lyap

        opt.zero_grad(set_to_none=True)
        L_total.backward()
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step()

        if not math.isfinite(L_task.item()) or not math.isfinite(gnorm):
            diverged = True
            if verbose: print(f"  [{arm} s{seed}] diverged at step {step}")
            break

        if step % cfg.lyap_every == 0:
            with torch.no_grad():
                pred = logits[:, cfg.copy_len + 1:].argmax(-1)
                gold = tgt[:, cfg.copy_len + 1:]
                acc = (pred == gold).float().mean().item()
                ch = (model.cell.phi.clamp_hits / model.cell.phi.clamp_total.clamp(min=1)).item()
            hist.append(dict(step=step, arm=arm, seed=seed,
                             loss=L_task.item(), acc=acc,
                             lam=lam.item(), gnorm=gnorm, clamp_rate=ch,
                             L_lyap=float(L_lyap.item()),
                             lyap_share=float(L_lyap.item() /
                                              max(abs(L_task.item()), 1e-8))))

    # ---- post-hoc: Benettin spectrum of the FULL recurrence and of phi alone
    model.eval()
    with torch.no_grad():
        inp, _ = make_copy_batch(cfg, 1, DEVICE, gen)
        xs = [model.emb(inp[0, t]) for t in range(inp.shape[1])]

    h0 = model.cell.init_state(1, DEVICE)[0].double()
    cellf = model.cell.double()

    sp_full = benettin_spectrum(lambda hh, x: cellf.step(hh.unsqueeze(0), x.unsqueeze(0).double())[0],
                                h0, [x.double() for x in xs], cfg.state_dim,
                                cfg.benettin_transient, cfg.benettin_measure)
    zero = torch.zeros(cfg.d_model, dtype=torch.float64, device=DEVICE)
    sp_phi = benettin_spectrum(lambda hh, x: cellf.phi(hh @ cellf.A.T.double()),
                               h0, [zero], cfg.state_dim,
                               cfg.benettin_transient, cfg.benettin_measure)

    r_learned = model.cell.phi.get_r().detach().cpu().numpy()

    summary = dict(
        arm=arm, seed=seed, diverged=diverged,
        wall_s=time.time() - t0,
        final_loss=hist[-1]["loss"] if hist else float("nan"),
        final_acc=hist[-1]["acc"] if hist else float("nan"),
        lam_mean=float(np.mean([h["lam"] for h in hist])) if hist else float("nan"),
        lam_final=hist[-1]["lam"] if hist else float("nan"),
        lam_frac_in_corridor=float(np.mean([(cfg.lam_low <= h["lam"] <= cfg.lam_high)
                                            for h in hist])) if hist else float("nan"),
        gnorm_max=float(np.max([h["gnorm"] for h in hist])) if hist else float("nan"),
        gnorm_median=float(np.median([h["gnorm"] for h in hist])) if hist else float("nan"),
        clamp_rate=hist[-1]["clamp_rate"] if hist else float("nan"),
        lyap_share_mean=float(np.mean([h["lyap_share"] for h in hist])) if hist else float("nan"),
        lam1_full=float(sp_full[0]), lam1_phi=float(sp_phi[0]),
        DKY_full=kaplan_yorke(sp_full), DKY_phi=kaplan_yorke(sp_phi),
        r_mean=float(r_learned.mean()), r_std=float(r_learned.std()),
    )
    return summary, pd.DataFrame(hist), sp_full.cpu().numpy()

## 5. Run — all arms × seeds

In [14]:
summaries, histories, spectra = [], [], {}

total = len(CFG.arms) * CFG.seeds
done = 0
for arm in CFG.arms:
    for seed in range(CFG.seeds):
        s, h, sp = train_one(arm, seed, CFG, verbose=True)
        summaries.append(s); histories.append(h); spectra[f"{arm}_s{seed}"] = sp
        done += 1
        print(f"[{done}/{total}] {arm} seed={seed} "
              f"acc={s['final_acc']:.3f} lam_mean={s['lam_mean']:+.4f} "
              f"DKY_full={s['DKY_full']:.2f} gmax={s['gnorm_max']:.1f} "
              f"{'DIVERGED' if s['diverged'] else ''}")

SUM = pd.DataFrame(summaries)
HIST = pd.concat(histories, ignore_index=True)
savetable(SUM, "phase0_summary.csv")
savetable(HIST, "phase0_history.csv")
savejson({k: v.tolist() for k, v in spectra.items()}, "phase0_spectra.json")
savejson({k: str(v) for k, v in asdict(CFG).items()}, "phase0_config.json")
SUM.groupby("arm")[["final_acc", "lam_mean", "lam_frac_in_corridor",
                    "DKY_full", "gnorm_max"]].mean()

[1/6] corridor seed=0 acc=0.062 lam_mean=-0.1481 DKY_full=0.00 gmax=6.4 
[2/6] corridor seed=1 acc=0.071 lam_mean=-0.2177 DKY_full=0.00 gmax=10.5 
[3/6] floss seed=0 acc=0.062 lam_mean=-0.1484 DKY_full=0.00 gmax=6.4 
[4/6] floss seed=1 acc=0.070 lam_mean=-0.2244 DKY_full=0.00 gmax=10.5 
[5/6] free seed=0 acc=0.063 lam_mean=-0.1417 DKY_full=0.00 gmax=6.4 
[6/6] free seed=1 acc=0.071 lam_mean=-0.1572 DKY_full=0.00 gmax=10.5 
saved /content/drive/My Drive/ChaoSSM/results/phase0/phase0_summary.csv
saved /content/drive/My Drive/ChaoSSM/results/phase0/phase0_history.csv
saved /content/drive/My Drive/ChaoSSM/results/phase0/phase0_spectra.json
saved /content/drive/My Drive/ChaoSSM/results/phase0/phase0_config.json


,final_acc,lam_mean,lam_frac_in_corridor,DKY_full,gnorm_max
arm,,,,,
corridor,0.066895,-0.182912,0.0,0.0,8.417717
floss,0.066406,-0.186371,0.0,0.0,8.417716
free,0.067383,-0.149482,0.0,0.0,8.417716


## 6. The four checks

### Check 1 — Trainability

In [15]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for arm in CFG.arms:
    d = HIST[HIST.arm == arm].groupby("step")[["loss", "acc", "gnorm"]].mean()
    ax[0].plot(d.index, d.loss, label=arm)
    ax[1].plot(d.index, d.acc, label=arm)
    ax[2].plot(d.index, d.gnorm, label=arm)
ax[0].set(xlabel="step", ylabel="task loss", title="Convergence")
ax[1].set(xlabel="step", ylabel="copy accuracy", title="Task accuracy")
ax[2].set(xlabel="step", ylabel="grad norm (pre-clip)", title="Gradient magnitude", yscale="log")
for a in ax: a.legend(); a.grid(alpha=.3)
fig.tight_layout(); savefig(fig, "phase0_check1_trainability.png"); plt.close(fig)

n_div = int(SUM.diverged.sum())
gmax = SUM.gnorm_max.max()
print(f"diverged runs : {n_div}/{len(SUM)}")
print(f"max grad norm : {gmax:.3g}")
print(f"CHECK 1 (trainability): {'PASS' if n_div == 0 and math.isfinite(gmax) else 'FAIL'}")

saved /content/drive/My Drive/ChaoSSM/results/phase0/phase0_check1_trainability.png
diverged runs : 0/6
max grad norm : 10.5
CHECK 1 (trainability): PASS


### Check 2 — Corridor controllability

**This is the decisive plot.** If the `corridor` distribution sits on top of `free`, the
regularizer is inert — the hinge is flat inside the corridor and the LM loss is choosing
λ̂ on its own. If `corridor` piles up against the lower edge (−0.05), the model is satisfying
the constraint while operating in the *stable* regime, not near criticality — the mechanism
is nominally present and functionally absent.

In [16]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for arm in CFG.arms:
    ax[0].hist(HIST[HIST.arm == arm].lam, bins=40, alpha=.5, label=arm, density=True)
for lo, hi, c in [(CFG.lam_low, CFG.lam_high, "green")]:
    ax[0].axvspan(lo, hi, color=c, alpha=.12, label="target corridor")
ax[0].axvline(0, color="k", ls=":", lw=1)
ax[0].set(xlabel=r"$\hat{\lambda}$", ylabel="density", title=r"Distribution of $\hat{\lambda}$ over training")
ax[0].legend(); ax[0].grid(alpha=.3)

for arm in CFG.arms:
    d = HIST[HIST.arm == arm].groupby("step").lam.agg(["mean", "std"])
    ax[1].plot(d.index, d["mean"], label=arm)
    ax[1].fill_between(d.index, d["mean"] - d["std"], d["mean"] + d["std"], alpha=.2)
ax[1].axhspan(CFG.lam_low, CFG.lam_high, color="green", alpha=.12)
ax[1].axhline(0, color="k", ls=":", lw=1)
ax[1].set(xlabel="step", ylabel=r"$\hat{\lambda}$", title=r"$\hat{\lambda}$ trajectory (mean $\pm$ sd over seeds)")
ax[1].legend(); ax[1].grid(alpha=.3)
fig.tight_layout(); savefig(fig, "phase0_check2_corridor.png"); plt.close(fig)

print(SUM.groupby("arm")[["lam_mean", "lam_final", "lam_frac_in_corridor"]]
      .agg(["mean", "std"]).round(4).to_string())

saved /content/drive/My Drive/ChaoSSM/results/phase0/phase0_check2_corridor.png
         lam_mean         lam_final         lam_frac_in_corridor     
             mean     std      mean     std                 mean  std
arm                                                                  
corridor  -0.1829  0.0492    -0.139  0.0182                  0.0  0.0
floss     -0.1864  0.0537    -0.160  0.0083                  0.0  0.0
free      -0.1495  0.0109    -0.146  0.0078                  0.0  0.0


In [17]:
# Paired comparison across seeds: is `corridor` distinguishable from `free`?
from scipy.stats import wilcoxon

def paired(a, b, col="lam_mean"):
    x = SUM[SUM.arm == a].sort_values("seed")[col].values
    y = SUM[SUM.arm == b].sort_values("seed")[col].values
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    if len(x) < 3 or np.allclose(x, y):
        return dict(pair=f"{a} vs {b}", n=len(x), stat=np.nan, p=np.nan,
                    median_diff=float(np.median(x - y)) if len(x) else np.nan)
    st, p = wilcoxon(x, y)
    return dict(pair=f"{a} vs {b}", n=len(x), stat=float(st), p=float(p),
                median_diff=float(np.median(x - y)))

rows = [paired("corridor", "free"), paired("corridor", "floss"), paired("floss", "free")]
stats_df = pd.DataFrame(rows)
print(stats_df.to_string(index=False))
savetable(stats_df, "phase0_check2_stats.csv")

pfree = stats_df.iloc[0].p
pmin = 2 / (2 ** CFG.seeds)
print(f"\nMinimum achievable two-sided p at n={CFG.seeds}: {pmin:.4f}")
if pmin > 0.05:
    print("CHECK 2: INCONCLUSIVE — too few seeds to reach significance at any effect size.")
elif not np.isfinite(pfree):
    print("CHECK 2: INCONCLUSIVE — test undefined (identical or too few finite pairs).")
elif pfree >= 0.05:
    print("CHECK 2: WARNING — `corridor` is NOT significantly different from `free`. "
          "The regularizer may be inert (barrier-not-attractor).")
else:
    print("CHECK 2: corridor arm is distinguishable from the unregularized control.")

# Direct magnitude test: is the regularizer contributing meaningfully to the loss?
share = SUM.groupby("arm").lyap_share_mean.mean()
print("\nMean L_Lyap / L_task by arm:")
print(share.round(6).to_string())
if share.get("corridor", 0) < 1e-3:
    print("\n!! The corridor penalty is <0.1% of the task loss. Even where lambda_hat sits\n"
          "   outside the corridor, the regularizer exerts almost no force. Raising alpha\n"
          "   (or redesigning the term) is required before the corridor means anything.")

             pair  n  stat   p  median_diff
 corridor vs free  2   NaN NaN    -0.033431
corridor vs floss  2   NaN NaN     0.003459
    floss vs free  2   NaN NaN    -0.036890
saved /content/drive/My Drive/ChaoSSM/results/phase0/phase0_check2_stats.csv

Minimum achievable two-sided p at n=2: 0.5000
CHECK 2: INCONCLUSIVE — too few seeds to reach significance at any effect size.

Mean L_Lyap / L_task by arm:
arm
corridor    0.000078
floss       0.000133
free        0.000000

!! The corridor penalty is <0.1% of the task loss. Even where lambda_hat sits
   outside the corridor, the regularizer exerts almost no force. Raising alpha
   (or redesigning the term) is required before the corridor means anything.


### Check 3 — Estimator validity

Two questions. Does the cheap power-iteration λ̂ agree with the Benettin reference? And does
λ of φ (the regularized quantity) agree with λ of the full recurrence (the quantity that
actually governs the dynamics)? A large gap on the second means the regularizer is steering
something other than what it is meant to control.

In [18]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(SUM.lam_mean, SUM.lam1_phi, c=[{"corridor":"C0","floss":"C1","free":"C2"}[a] for a in SUM.arm])
lims = [min(SUM.lam_mean.min(), SUM.lam1_phi.min()), max(SUM.lam_mean.max(), SUM.lam1_phi.max())]
ax[0].plot(lims, lims, "k--", lw=1)
ax[0].set(xlabel=r"power-iteration $\hat{\lambda}$ (training mean)",
          ylabel=r"Benettin $\lambda_1(\phi)$", title="Cheap estimator vs. reference")
ax[0].grid(alpha=.3)

ax[1].scatter(SUM.lam1_phi, SUM.lam1_full,
              c=[{"corridor":"C0","floss":"C1","free":"C2"}[a] for a in SUM.arm])
lims2 = [min(SUM.lam1_phi.min(), SUM.lam1_full.min()), max(SUM.lam1_phi.max(), SUM.lam1_full.max())]
ax[1].plot(lims2, lims2, "k--", lw=1)
ax[1].set(xlabel=r"$\lambda_1(\phi)$  (what S4.4 regularizes)",
          ylabel=r"$\lambda_1$(full recurrence)  (what governs dynamics)",
          title="Regularized quantity vs. actual dynamics")
ax[1].grid(alpha=.3)
fig.tight_layout(); savefig(fig, "phase0_check3_estimators.png"); plt.close(fig)

corr_est = SUM[["lam_mean", "lam1_phi"]].corr().iloc[0, 1]
gap = (SUM.lam1_full - SUM.lam1_phi).abs()
print(f"corr(power-iteration, Benettin phi) = {corr_est:.3f}")
print(f"|lambda_full - lambda_phi|: median={gap.median():.4f}  max={gap.max():.4f}")
print(SUM.groupby('arm')[['lam_mean','lam1_phi','lam1_full']].mean().round(4).to_string())

saved /content/drive/My Drive/ChaoSSM/results/phase0/phase0_check3_estimators.png
corr(power-iteration, Benettin phi) = -0.775
|lambda_full - lambda_phi|: median=0.2462  max=0.2800
          lam_mean  lam1_phi  lam1_full
arm                                    
corridor   -0.1829   -0.7286    -0.4827
floss      -0.1864   -0.7288    -0.4826
free       -0.1495   -0.7278    -0.4827


### Check 4 — Non-degeneracy

`D_KY > 1` on the full recurrence is the evidence that the learned system is a genuine
strange attractor rather than a fixed point or a near-linear map. The proposal predicts
linear SSMs sit at D_KY ≈ 1 (consistent with Halloran et al.'s λ ≤ 0 result), so this is
the quantity that has to separate.

The learned `r` distribution is the second half of the check: if `r` collapses toward 3.57
(the bottom of the parameterized range) the map is drifting out of the chaotic regime by
choice.

In [19]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
SUM.boxplot(column="DKY_full", by="arm", ax=ax[0]); ax[0].axhline(1.0, color="r", ls="--")
ax[0].set(title=r"$D_{KY}$ (full recurrence)", xlabel="arm", ylabel=r"$D_{KY}$")

for arm in CFG.arms:
    sub = [spectra[k] for k in spectra if k.startswith(arm)]
    if sub:
        m = np.mean(np.stack(sub), axis=0)
        ax[1].plot(m, marker=".", label=arm)
ax[1].axhline(0, color="k", ls=":"); ax[1].legend(); ax[1].grid(alpha=.3)
ax[1].set(xlabel="index i", ylabel=r"$\lambda_i$", title="Lyapunov spectrum (full recurrence)")

SUM.boxplot(column="r_mean", by="arm", ax=ax[2])
ax[2].axhline(3.57, color="r", ls="--", label="chaos onset")
ax[2].set(title="learned r (mean)", xlabel="arm", ylabel="r"); ax[2].legend()
fig.suptitle(""); fig.tight_layout(); savefig(fig, "phase0_check4_degeneracy.png"); plt.close(fig)

print(SUM.groupby("arm")[["DKY_full", "DKY_phi", "r_mean", "clamp_rate"]].mean().round(3).to_string())
frac_chaotic = (SUM.DKY_full > 1.0).mean()
print(f"\nruns with D_KY(full) > 1: {frac_chaotic:.0%}")
if SMOKE_TEST:
    print("CHECK 4: NOT MEANINGFUL under SMOKE_TEST (too few steps to learn anything). "
          "Re-run with SMOKE_TEST=False.")
else:
    print(f"CHECK 4: {'PASS' if frac_chaotic > 0.5 else 'FAIL — dynamics are degenerate'}")
print(f"\nmean clamp activation rate: {SUM.clamp_rate.mean():.1%}  "
      "(high values mean the state is routinely leaving (0,1) and being clipped, "
      "which flattens the map's Jacobian and biases lambda downward)")

saved /content/drive/My Drive/ChaoSSM/results/phase0/phase0_check4_degeneracy.png
          DKY_full  DKY_phi  r_mean  clamp_rate
arm                                            
corridor       0.0      0.0   3.785       0.695
floss          0.0      0.0   3.785       0.695
free           0.0      0.0   3.785       0.695

runs with D_KY(full) > 1: 0%
CHECK 4: NOT MEANINGFUL under SMOKE_TEST (too few steps to learn anything). Re-run with SMOKE_TEST=False.

mean clamp activation rate: 69.5%  (high values mean the state is routinely leaving (0,1) and being clipped, which flattens the map's Jacobian and biases lambda downward)


## 7. Verdict

Read the checks together, not individually.

| Outcome | Reading | What to do |
|---|---|---|
| All four pass, `corridor` separates from `free`, D_KY > 1 | The premise holds. | Proceed to the full study; the corridor design is sound. |
| Checks 1/3/4 pass but Check 2 shows `corridor` ≈ `free` | The hinge is a barrier, not an attractor — the constraint is satisfied vacuously. | **Redesign the regularizer before spending compute.** Options: a soft attractor toward the corridor midpoint, an explicit reward for λ̂ > 0, or a scheduled target. This is a finding, not a failure. |
| Check 2 shows λ̂ pinned at the lower edge | The LM objective actively prefers the stable regime. | The expressivity argument in §3.2 needs empirical defense before it can carry a paper. |
| Check 4 fails (D_KY ≈ 1) | The learned map is not chaotic in practice. | "Chaos" is decorative; the architecture reduces to a nonlinear SSM and the novelty claim in §8 weakens badly. |
| Check 3 shows λ(φ) ≫ λ(full) | The regularizer controls the wrong quantity. | Move the regularizer to the full recurrence Jacobian — a correction to §4.4. |
| Check 1 fails | Backprop-through-chaos is intractable at these settings. | Revisit `bptt_k`, the Lyapunov-aware clipping schedule, and the warmup in §4.6. |

Any of these outcomes is worth knowing at a cost of hours rather than thousands of GPU-hours.
The second row is the one I would bet on, and it is entirely fixable — but only if it is
caught now rather than in month three.

In [20]:
verdict = dict(
    smoke_test=SMOKE_TEST,
    n_runs=len(SUM),
    check1_trainability="PASS" if int(SUM.diverged.sum()) == 0 else "FAIL",
    check2_corridor_vs_free_p=float(stats_df.iloc[0].p),
    check2_lam_mean_by_arm=SUM.groupby("arm").lam_mean.mean().round(4).to_dict(),
    check3_estimator_corr=float(corr_est),
    check3_phi_vs_full_median_gap=float(gap.median()),
    check4_frac_DKY_gt_1=float(frac_chaotic),
    regularizer_share_of_loss=SUM.groupby("arm").lyap_share_mean.mean().round(8).to_dict(),
    clamp_activation_rate=float(SUM.clamp_rate.mean()),
    check4_DKY_by_arm=SUM.groupby("arm").DKY_full.mean().round(3).to_dict(),
)
savejson(verdict, "phase0_verdict.json")
print(json.dumps(verdict, indent=2))

saved /content/drive/My Drive/ChaoSSM/results/phase0/phase0_verdict.json
{
  "smoke_test": true,
  "n_runs": 6,
  "check1_trainability": "PASS",
  "check2_corridor_vs_free_p": NaN,
  "check2_lam_mean_by_arm": {
    "corridor": -0.1829,
    "floss": -0.1864,
    "free": -0.1495
  },
  "check3_estimator_corr": -0.7751318291284245,
  "check3_phi_vs_full_median_gap": 0.2461507174278542,
  "check4_frac_DKY_gt_1": 0.0,
  "regularizer_share_of_loss": {
    "corridor": 7.776e-05,
    "floss": 0.00013342,
    "free": 0.0
  },
  "clamp_activation_rate": 0.694841722647349,
  "check4_DKY_by_arm": {
    "corridor": 0.0,
    "floss": 0.0,
    "free": 0.0
  }
}
